In [ ]:
# Crawl outputs of open-gira TC/grid analysis, specifically:
# power/by_storm_set/{STORM_SET}/disruption/pop_affected_RP/{RETURN_PERIOD}.gpq files
# for each return period, plot a bar chart of calibrated (variable threshold) of 1 in N year
# disruption events by region, by country.

In [ ]:
from collections import defaultdict
import os

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Input data paths
country_thresholds_path = "../data/country_thresholds.csv"
wb_income_classification_path = "../data/wb_income_classification.csv"
targets_path = "/data/tc-grid/open-gira/results/power/targets.geoparquet"
results_dir = "/data/tc-grid/open-gira/results"

# Output paths
plot_dir = "../plots"

# Return periods to process and plot
return_periods = [10, 20, 50, 100, 200, 500]
# Mapping from region display name to region bounding box
regions = {
    "Americas": [-103, 4, -48, 32],
    "Indian Ocean": [20, -35, 95, 27],
    "West Pacific": [101.5, -55, 170, 60],
}
# Mapping from open-gira STORM_SET to display name
atmospheres = {
    "STORM_constant": "2020 STORM",
    "STORM_CNRM-CM6-1-HR": "2050 STORM RCP8.5 CNRM",
    "STORM_CMCC-CM2-VHR4": "2050 STORM RCP8.5 CMCC",
    "STORM_EC-Earth3P-HR": "2050 STORM RCP8.5 EC-Earth",
    "STORM_HadGEM3-GC31-HM": "2050 STORM RCP8.5 HadGEM3",
    "IRIS_PRESENT": "2020 IRIS",
    "IRIS_SSP5-2050": "2050 IRIS SSP5-8.5",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2010": "2010 CHAZ UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2010": "2010 CHAZ CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2010": "2010 CHAZ CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2010": "2010 CHAZ EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2010": "2010 CHAZ MIROC6",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2050": "2050 CHAZ SSP5-8.5 UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2050": "2050 CHAZ SSP5-8.5 CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2050": "2050 CHAZ SSP5-8.5 CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2050": "2050 CHAZ SSP5-8.5 EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2050": "2050 CHAZ SSP5-8.5 MIROC6",
    "emanuel_ssp-585_gcm-cesm2_epoch-2005": "2005 Emanuel CESM2 2005",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2005": "2005 Emanuel CNRM6 2005",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2005": "2005 Emanuel EC-Earth6 2005",
    "emanuel_ssp-585_gcm-fgoals_epoch-2005": "2005 Emanuel FGOALS 2005",
    "emanuel_ssp-585_gcm-ipsl6_epoch-2005": "2005 Emanuel IPSL6 2005",
    "emanuel_ssp-585_gcm-miroc6_epoch-2005": "2005 Emanuel MIROC6 2005",
    "emanuel_ssp-585_gcm-mpi6_epoch-2005": "2005 Emanuel MPI6 2005",
    "emanuel_ssp-585_gcm-mri6_epoch-2005": "2005 Emanuel MRI6 2005",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2005": "2005 Emanuel UKMO6 2005",
    "emanuel_ssp-585_gcm-cesm2_epoch-2050": "2050 Emanuel CESM2 2050",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2050": "2050 Emanuel CNRM6 2050",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2050": "2050 Emanuel EC-Earth6 2050",
    "emanuel_ssp-585_gcm-fgoals_epoch-2050": "2050 Emanuel FGOALS 2050",
    "emanuel_ssp-585_gcm-ipsl6_epoch-2050": "2050 Emanuel IPSL6 2050",
    "emanuel_ssp-585_gcm-miroc6_epoch-2050": "2050 Emanuel MIROC6 2050",
    "emanuel_ssp-585_gcm-mpi6_epoch-2050": "2050 Emanuel MPI6 2050",
    "emanuel_ssp-585_gcm-mri6_epoch-2050": "2050 Emanuel MRI6 2050",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2050": "2050 Emanuel UKMO6 2050",
}
# Specify group members (GCM variants) of track sets
gcm_groups = {
    "IRIS baseline": ["2020 IRIS"],
    "IRIS SSP5-8.5 2050": ["2050 IRIS SSP5-8.5"],
    "STORM baseline": ["2020 STORM"],
    "STORM RCP8.5 2050": [
        "2050 STORM RCP8.5 CNRM",
        "2050 STORM RCP8.5 CMCC",
        "2050 STORM RCP8.5 EC-Earth",
        "2050 STORM RCP8.5 HadGEM3"
    ],
    "CHAZ baseline": [
        "2010 CHAZ UKESM1",
        "2010 CHAZ CESM2",
        "2010 CHAZ CNRM",
        "2010 CHAZ EC-Earth3",
        "2010 CHAZ MIROC6",
    ],
    "CHAZ SSP5-8.5 2050": [
        "2050 CHAZ SSP5-8.5 UKESM1",
        "2050 CHAZ SSP5-8.5 CESM2",
        "2050 CHAZ SSP5-8.5 CNRM",
        "2050 CHAZ SSP5-8.5 EC-Earth3",
        "2050 CHAZ SSP5-8.5 MIROC6",
    ],   
    "Emanuel baseline": [
        "2005 Emanuel CESM2 2005",
        "2005 Emanuel CNRM6 2005",
        "2005 Emanuel EC-Earth6 2005",
        "2005 Emanuel FGOALS 2005",
        "2005 Emanuel IPSL6 2005",
        "2005 Emanuel MIROC6 2005",
        "2005 Emanuel MPI6 2005",
        "2005 Emanuel MRI6 2005",
        "2005 Emanuel UKMO6 2005",
    ],
    "Emanuel SSP5-8.5 2050": [
        "2050 Emanuel CESM2 2050",
        "2050 Emanuel CNRM6 2050",
        "2050 Emanuel EC-Earth6 2050",
        "2050 Emanuel FGOALS 2050",
        "2050 Emanuel IPSL6 2050",
        "2050 Emanuel MIROC6 2050",
        "2050 Emanuel MPI6 2050",
        "2050 Emanuel MRI6 2050",
        "2050 Emanuel UKMO6 2050",
    ],
}
colours = {
    "IRIS baseline": "darkgreen",
    "IRIS SSP5-8.5 2050": "lightgreen",
    "STORM baseline": "purple",
    "STORM RCP8.5 2050": "plum",
    "CHAZ baseline": "red",
    "CHAZ SSP5-8.5 2050": "pink",
    "Emanuel baseline": "blue",
    "Emanuel SSP5-8.5 2050": "lightblue",
}

In [ ]:
def lookup(df: pd.DataFrame, rows, cols) -> np.ndarray:
    """
    Helper function to lookup values based on list of `rows` and `cols` labels.
    Does this really not exist in pandas already?!

    Args:
        df: Table to lookup from
        rows: Iterable of row indicies to select
        cols: Iterable of column indicies to select (same length as `rows`)

    Returns:
        Elements of `df` that are indexed by `rows` and `cols`.
    """
    return df.to_numpy()[df.index.get_indexer(rows), df.columns.get_indexer(cols)]

In [ ]:
def process_disruption_data(
    region: str,
    region_bbox: tuple[float, float, float, float],
    return_period: int,
    atmospheres: dict,
    default_threshold: float,
    thresholds_by_country: pd.DataFrame,
    results_dir: str,
    gcm_groups: dict[str, list[str]],
) -> pd.DataFrame:
    """
    Read in disruption figures from disk
    Select appropriate threshold for each country
    Calculate min, mean and max for any multi-GCM track sets

    Args:
        region: Region name, maps to bounding box in global
        region_bbox: Bounding box for region in decimal degrees: xmin, ymin, xmax, ymax
        return_period: Return period of disruption event in years to lookup
        atmospheres: Mapping from open-gira STORM_SET to human readable name
        default_threshold: Threshold to use if we haven't set via another means
        thresholds_by_country: Threshold values to use per-country
        results_dir: Path to open-gira/results folder
        gcm_groups: Mapping from aggregated scenario label to list of member scenario
            names (as they appear in atmospheres values). For each group, individual
            GCM runs are collapsed into a single row with mean/min/max across models.
            Example:
                {
                    "2010 CHAZ": ["2010 CHAZ UKESM1", "2010 CHAZ CESM2", ...],
                    "2050 CHAZ SSP5-8.5": ["2050 CHAZ SSP5-8.5 UKESM1", ...],
                }

    Returns:
        Processed disruption data for region, indexed by country ISO A3 code and scenario.
        Multi-GCM track sets are represented by their cross-model mean, with min/max
        error bar offsets in pop_at_risk_min and pop_at_risk_max respectively.
    """

    # read in data from disk
    data_by_atmosphere = []
    for atmosphere in atmospheres.keys():
        path = os.path.join(results_dir, f"power/by_storm_set/{atmosphere}/disruption/pop_affected_RP/{return_period}.gpq")
        df = gpd.read_parquet(path)

        # extend our dataframe to have countries from thresholds_by_country
        # data will be null for these rows, but it allows us to index more easily later
        missing_indicies = thresholds_by_country.index.difference(df.index)
        df = pd.concat([df, pd.DataFrame(index=missing_indicies)])

        # set countries with no calibrated value to the global default
        df["pop_at_risk"] = df[f"{default_threshold:.1f}"]
        # using the thresholds_by_country mapping, lookup population at risk for calibrated countries
        df.loc[thresholds_by_country.index, "pop_at_risk"] = lookup(
            df,
            thresholds_by_country.index,
            thresholds_by_country["threshold_ms-1"].values.astype(str)  # column labels are stringified 1 d.p. floats
        ).astype(float)
        # drop the now redundant master set of thresholds
        df = df.loc[:, ["pop_at_risk", "NAME_0", "geometry"]].sort_values("pop_at_risk", ascending=False)
        df["ATMOSPHERE"] = atmospheres[atmosphere]
        df = df.reset_index().set_index(["ISO_A3", "ATMOSPHERE"])
        data_by_atmosphere.append(df)

    data = pd.concat(data_by_atmosphere)

    # init columns for error bars
    data["pop_at_risk_min"] = 0
    data["pop_at_risk_max"] = 0

    # filter by region
    xmin, ymin, xmax, ymax = region_bbox
    data = data.cx[xmin:xmax, ymin:ymax]

    # filter out smallest countries, sorry
    mask = data.loc[:, "pop_at_risk"].groupby("ISO_A3").max() > 1E6
    notable_countries = mask[mask].index.values
    data = data[data.index.get_level_values(0).isin(notable_countries)]
    data = data.drop(columns=["NAME_0", "geometry"])

    # for each multi-GCM group, collapse individual model runs into a mean row
    # with min/max offsets for error bars
    aggregated_groups = []
    all_gcm_member_scenarios: set[str] = set()

    for group_label, member_scenarios in gcm_groups.items():
        all_gcm_member_scenarios.update(member_scenarios)

        subset = data[data.index.get_level_values("ATMOSPHERE").isin(member_scenarios)]
        groupby_country = subset.loc[:, "pop_at_risk"].groupby("ISO_A3")

        mean = groupby_country.mean()
        minimum = mean - groupby_country.min()
        minimum.name = "pop_at_risk_min"
        maximum = groupby_country.max() - mean
        maximum.name = "pop_at_risk_max"

        group_df = pd.DataFrame(
            index=pd.MultiIndex.from_product(
                [mean.index, [group_label]], names=["ISO_A3", "ATMOSPHERE"]
            )
        )
        group_df = group_df.join(mean).join(minimum).join(maximum)
        aggregated_groups.append(group_df)

    # replace individual GCM member rows with their aggregated group rows
    data = data[~data.index.get_level_values("ATMOSPHERE").isin(all_gcm_member_scenarios)]
    data = pd.concat([data] + aggregated_groups)

    data.index.names = ["Country", "Scenario"]

    return data

In [ ]:
def plot_percent_and_absolute(
    return_period: int,
    region_data: dict[pd.DataFrame],
    connected_pop: pd.DataFrame,
    plot_dir: str,
    order_by: str,
) -> None:
    """
    Plot bar charts (absolute and %) of population at risk of disconnection for all regions

    Args:
        return_period: Return period of disruption event in years to lookup
        region_data: Mapping from region name to table of disruption data indexed by country and scenario
        connected_pop: Table of connected population, indexed by country
        plot_dir: Directory to save plots to
        order_by: Scenario label to order results by
    """
    plot_kwds = {
        "kind": "bar",
        "alpha": 0.7,
        "color": colours,
        "width": 0.85,
        "legend": False,
        "error_kw": {'elinewidth': 0.5},
    }
    spine_width = 0.5
    label_size = 5
    grid_alpha = 0.15

    for i, (region, df) in enumerate(region_data.items()):
        f, axes = plt.subplots(2, 1, height_ratios=[3,2])
        abs_ax, rel_ax = axes

        n_atmospheric_states = len(df.index.get_level_values(1).unique())
    
        # calculate percentage population at risk for later
        perc = 100 * df / connected_pop
        
        # find fraction of connected population at risk in each case 
        df = df.unstack().sort_values([("pop_at_risk", order_by)])
        df = df.sort_index(axis=1, level=0, ascending=False)
        # shape (scenario, 2, country), with the second dimension being [mean - min, max - mean]
        df_yerr = np.hstack([df.pop_at_risk_min.values[:, np.newaxis], df.pop_at_risk_max.values[:, np.newaxis]]).T
        
        df.pop_at_risk.plot(
            ax=abs_ax,
            yerr=df_yerr,
            **plot_kwds,
        )
        abs_ax.set_ylim(7E4, 1.4E8)
        abs_ax.set_yscale("log")
        abs_ax.set_xticklabels([])
        abs_ax.set_xlabel("")
        abs_ax.xaxis.set_tick_params(labelsize=label_size)
        abs_ax.yaxis.set_tick_params(labelsize=label_size)
        abs_ax.grid(which="both", alpha=grid_alpha)
        abs_ax.text(
            0,
            6E7,
            region,
            fontsize=10,
            horizontalalignment='left',
            verticalalignment='top',
        )
        abs_ax.set_zorder(1)
        
        perc = perc.unstack().sort_values([("pop_at_risk", order_by)])
        perc = perc.sort_index(axis=1, level=0, ascending=False)
        # we want the absolute and percentage plots to have the same members and ordering, so index with df.index
        perc_with_abs_order = perc.loc[df.index, :]
        perc_yerr = np.hstack(
            [
                perc_with_abs_order.pop_at_risk_min.values[:, np.newaxis],
                perc_with_abs_order.pop_at_risk_max.values[:, np.newaxis]
            ]
        ).T
        perc_with_abs_order.pop_at_risk.plot(
            ax=rel_ax,
            yerr=perc_yerr,
            **plot_kwds,
        )
        
        rel_ax.grid(which="both", alpha=grid_alpha)
        rel_ax.xaxis.set_tick_params(labelsize=label_size)
        rel_ax.yaxis.set_tick_params(labelsize=label_size)
        rel_ax.set_xlabel("", labelpad=0, fontsize=label_size)
        rel_ax.set_ylim(0, 119)
    
        abs_ax.set_ylabel("Population affected", labelpad=12, fontsize=label_size)
        rel_ax.set_ylabel("Population affected\n[% of connected population]", labelpad=7, fontsize=label_size)
    
        rel_ax.legend(
            loc="upper right",
            bbox_to_anchor=(1, 1),
            ncols=n_atmospheric_states // 2,
            fontsize=4,
            framealpha=1,
        )
        rel_ax.set_zorder(1)
        rel_ax.yaxis.set_ticks_position('both')
        abs_ax.yaxis.set_ticks_position('both')
        
        plt.setp(rel_ax.spines.values(), linewidth=spine_width)
        rel_ax.xaxis.set_tick_params(width=spine_width)
        rel_ax.yaxis.set_tick_params(width=spine_width)
        plt.setp(abs_ax.spines.values(), linewidth=spine_width)
        abs_ax.xaxis.set_tick_params(width=spine_width)
        abs_ax.yaxis.set_tick_params(width=spine_width)
    
        f.subplots_adjust(hspace=0)
        plt.subplots_adjust(left=0.15, right=0.93, top=0.95, wspace=0.03)
        
        filename = os.path.join(plot_dir, f"pop-perc-abs_region-{region.lower().replace(' ', '-')}_rp-{return_period}.pdf")
        f.savefig(filename, dpi=254)
        plt.close(f)

In [ ]:
# estimate population connected to grid in each country
targets = pd.read_parquet(targets_path, columns=["population", "iso_a3"])
connected_pop = targets.groupby("iso_a3").sum()
connected_pop.index.name = "Country"
connected_pop = connected_pop.rename(columns={"population": "pop_at_risk"})
connected_pop["pop_at_risk_min"] = connected_pop["pop_at_risk"]
connected_pop["pop_at_risk_max"] = connected_pop["pop_at_risk"]

# assemble optimum country to threshold mapping
# calibrated thresholds, ms-1
per_country_calibrated_thresholds = pd.read_csv(country_thresholds_path).rename(columns={"iso_a3": "ISO_A3"}).set_index("ISO_A3")
# thresholds based on income, ms-1
income_threshold_map = {"L": "30.0", "LM": "30.0", "UM": "30.0", "H": "32.5"}
thresholds = pd.read_csv(wb_income_classification_path, comment="#").set_index("ISO_A3")
thresholds["threshold_ms-1"] = thresholds.INCOME_LEVEL.map(income_threshold_map)
thresholds.loc[per_country_calibrated_thresholds.index, "threshold_ms-1"] = per_country_calibrated_thresholds["threshold_ms-1"].astype(str)
# global default threshold, ms-1
default_threshold = 30.0

In [ ]:
# read and process data, store in `region_data`
region_data = defaultdict(dict)
for region, region_bbox in regions.items():
    for return_period in return_periods:
        print(region, return_period)
        region_data[region][return_period] = process_disruption_data(
            region,
            region_bbox,
            return_period,
            atmospheres,
            default_threshold,
            thresholds,
            results_dir,
            gcm_groups
        )

In [ ]:
# plot bar chart of 1 in N year disruption event, by region, by country
for return_period in return_periods:
    to_plot = {}
    for region in regions.keys():
        to_plot[region] = region_data[region][return_period]
    plot_percent_and_absolute(return_period, to_plot, connected_pop, plot_dir, "STORM baseline")